# Craigslist used-car price modeling — results and interpretation

**Purpose:** Summarize the **time-aware** price model trained in GCP (`train-dt` Cloud Function) on **enriched listings** (regex + LLM ETL → `listings_master_llm.csv`).

**This notebook does not retrain** and **does not use GCP credentials**. It only reads artifacts synced into `results/` by GitHub Actions (**Sync model artifacts to repo**).

**Metrics** (MAE, MAPE, RMSE, Bias) are computed on the **holdout day** in **original dollars** (`actual_price` vs `pred_price`), even when the model was fit in log space (`expm1` applied for reporting).

## 1. Data and features (high level)

- **Source:** Craigslist car listings scraped to GCS; **regex** structured extract + **Vertex Gemini** enrichment; master table **`listings_master_llm.csv`**.
- **Key enriched fields** used in the model include make/model, mileage, year, transmission, fuel, drive, condition, title status, body type, seller type, location (city/state), and **`zip_prefix`** from a cleaned 5-digit ZIP string (conservative nulling when invalid).
- **Training filters** (applied before fitting): valid positive price in a bounded range, plausible model year vs scrape year, plausible mileage, exclusion of clearly non-standard **title** rows (e.g. salvage / parts-only / missing) and **condition = project**. Exact drop counts appear in **`metrics.json`** under **`filtering`**.
- **Numeric features** are intentionally compact: **`vehicle_age`**, **`log_mileage`**, **`cylinders`**, sometimes plus **`year_num`** / **`mileage_num`** if a small validation check shows clear gain (**feature variant A vs B** in synced metadata).

## 2. Modeling strategy

- **Time-aware split (no random holdout for grading):** listings are grouped by **`America/New_York` local calendar date** from `scraped_at`. The model trains on **all days strictly before the latest day** in the dataset; the **latest day is the final holdout** (“today” in the data).
- **Validation for model choice:** when **at least three** distinct local dates exist, the **second-latest** day is used as a **validation** day. All candidate models are compared on that same day so selection is **fair and forward-looking**, not i.i.d. random split.
- **Candidates:** `DecisionTreeRegressor`, `RandomForestRegressor`, `ExtraTreesRegressor`, `HistGradientBoostingRegressor` (sklearn only).
- **Targets compared:** **`price_num` (raw USD)** vs **`log1p(price_num)`**. Log models report errors in dollars after **`numpy.expm1`** on predictions.
- **Tuning:** After a default-parameter benchmark, only the **top two** **(model, target)** pairs are tuned with **`ParameterSampler`** on the **same** validation day. The **winner** is refit on **all** pre-holdout training rows, then evaluated once on the **latest** holdout day.

**Why not random K-fold?** Craigslist prices drift by week and listing mix; a random split can leak future-like patterns and overstate accuracy. A **calendar** validation and holdout better match real deployment.

In [ ]:
# Repo root: Colab shallow-clone, or local checkout with ./results
import os
import sys

DEFAULT_REPO = "https://github.com/OPIM5512-mjb24001/myscrapers-mjb24001-v3.git"
REPO_URL = os.environ.get("NOTEBOOK_REPO_URL", DEFAULT_REPO)
BRANCH = os.environ.get("NOTEBOOK_REPO_BRANCH", "main")

if "google.colab" in sys.modules:
    import subprocess

    subprocess.run(["rm", "-rf", "repo"], check=False)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "repo"],
        check=True,
    )
    os.chdir("repo")
    print(f"Cloned: {REPO_URL} (branch {BRANCH})")
else:
    if os.path.isdir("results"):
        print("Using current directory as repo root (./results found).")
    elif os.path.isdir(os.path.join("..", "results")):
        os.chdir("..")
        print("Using parent directory as repo root.")
    else:
        print(
            "Tip: run from the repository root so ./results exists, or use Colab to clone."
        )

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display, Image, Markdown
except ImportError:
    display = print
    Image = None

    def Markdown(x):
        print(x)


ROOT = Path(".").resolve()
RESULTS = ROOT / "results"
METRICS_DIR = RESULTS / "metrics"


def latest_run_id() -> str | None:
    if not METRICS_DIR.is_dir():
        return None
    files = sorted(METRICS_DIR.glob("*-metrics.json"))
    if not files:
        return None
    return files[-1].stem.replace("-metrics", "")


run_id = latest_run_id()
metrics_path = METRICS_DIR / f"{run_id}-metrics.json" if run_id else None
model_info_path = METRICS_DIR / f"{run_id}-model_info.json" if run_id else None
bench_csv_path = METRICS_DIR / f"{run_id}-model_benchmark.csv" if run_id else None

metrics = {}
if metrics_path and metrics_path.is_file():
    try:
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Could not parse metrics JSON: {e}")
        metrics = {}

if run_id:
    display(Markdown(f"**Latest synced run_id:** `{run_id}`"))
else:
    display(
        Markdown(
            "_No `results/metrics/*-metrics.json` found. Train (not dry_run) and run **Sync model artifacts**."
        )
    )

## 3. Model comparison (validation day)

The table below uses **`model_benchmark.csv`** when synced (flat rows: default benchmark + tuned finalists). **`selected=yes`** marks the configuration that was refit on all pre-holdout data. If the CSV is missing (older runs), we fall back to **`metrics.json` → `benchmark`**.

In [ ]:
bench_df = pd.DataFrame()

if bench_csv_path and bench_csv_path.is_file():
    try:
        bench_df = pd.read_csv(bench_csv_path)
    except Exception as e:
        print(f"Could not read model_benchmark.csv: {e}")

if bench_df.empty and metrics.get("benchmark"):
    b = metrics["benchmark"]
    rows = []
    for r in b.get("candidate_results") or []:
        rows.append(
            {
                "stage": "benchmark_default_params",
                "model": r.get("model"),
                "target_strategy": r.get("target_strategy"),
                "val_mae": r.get("val_mae"),
                "val_rmse": r.get("val_rmse"),
                "val_mape": r.get("val_mape"),
                "val_bias": r.get("val_bias"),
                "selected": "no",
            }
        )
    chosen = metrics.get("chosen_model")
    ts = metrics.get("target_strategy")
    for f in b.get("tuned_finalists") or []:
        m = f.get("validation_after_tune") or {}
        win = f.get("model") == chosen and f.get("target_strategy") == ts
        rows.append(
            {
                "stage": "tuned_finalist",
                "model": f.get("model"),
                "target_strategy": f.get("target_strategy"),
                "val_mae": m.get("val_mae"),
                "val_rmse": m.get("val_rmse"),
                "val_mape": m.get("val_mape"),
                "val_bias": m.get("val_bias"),
                "selected": "yes" if win else "no",
            }
        )
    bench_df = pd.DataFrame(rows)

if bench_df.empty:
    print("No benchmark table available for this run.")
else:
    display(bench_df.sort_values(["stage", "val_mae"], na_position="last"))

if metrics.get("model_comparison"):
    display(Markdown("**Selection rule (from metrics):**"))
    display(Markdown(f"> {metrics['model_comparison'].get('selection_rule', '')}"))

if metrics.get("benchmark", {}).get("skip_reason"):
    display(
        Markdown(
            f"_Note: benchmark skipped or limited — `{metrics['benchmark']['skip_reason']}`._"
        )
    )

### Interpretation (comparison)

- **Lower validation MAE (USD)** on the calendar validation day is the primary ranking signal; **RMSE** penalizes large errors more; **bias** shows systematic over/under-pricing.
- **Raw vs log:** Log targets often stabilize variance when prices are skewed; the pipeline picks the strategy that validates better on **dollars**, not in log space.
- **Forests vs boosting:** Tree ensembles differ in bias–variance tradeoffs; the benchmark makes the choice **explicit** rather than defaulting to a single algorithm.

## 4. Final holdout performance (latest run)

These values come from **`metrics.json`** for the latest synced run (same numbers as **`metrics.csv`**).

In [ ]:
if metrics:
    hold = pd.DataFrame(
        [
            {
                "eval_date_local": metrics.get("eval_date_local"),
                "train_rows": metrics.get("train_rows"),
                "holdout_rows": metrics.get("holdout_rows"),
                "MAE": metrics.get("mae"),
                "MAPE_%": metrics.get("mape"),
                "RMSE": metrics.get("rmse"),
                "Bias": metrics.get("bias"),
                "chosen_model": metrics.get("chosen_model"),
                "target_strategy": metrics.get("target_strategy"),
                "target_transform": metrics.get("target_transform"),
            }
        ]
    )
    display(hold)
else:
    print("No metrics.json loaded.")

### Row filtering (latest run)

Counts come from **`metrics.json` → `filtering`** (same rules as training).

In [ ]:
fl = metrics.get("filtering") or {}
if fl:
    fd = pd.DataFrame([{"metric": k, "value": v} for k, v in fl.items()])
    display(fd)
else:
    print("No filtering block in metrics.json.")

## 5. Metric trends over training runs

**`results/history/metrics_history.csv`** appends one row per training job. Use **`timestamp_utc`** or **`run_id`** on the x-axis. Duplicate `run_id` rows can appear if the same run was synced twice — treat the file as operational telemetry rather than a strict experiment log.

In [ ]:
hist = pd.DataFrame()
hist_path = RESULTS / "history" / "metrics_history.csv"

if not RESULTS.is_dir():
    print("No results/ folder.")
elif not hist_path.is_file():
    print("No metrics_history.csv yet.")
else:
    try:
        hist = pd.read_csv(hist_path)
        if "timestamp_utc" in hist.columns:
            hist["timestamp_utc"] = pd.to_datetime(
                hist["timestamp_utc"], utc=True, errors="coerce"
            )
        display(hist.tail(8))
    except Exception as e:
        print(f"Could not load history: {e}")

if hist.empty:
    print("Skip trend plots.")
else:
    if "timestamp_utc" in hist.columns and hist["timestamp_utc"].notna().any():
        x = hist["timestamp_utc"]
        xlabel = "timestamp_utc"
    elif "run_id" in hist.columns:
        x = hist["run_id"].astype(str)
        xlabel = "run_id"
    elif "eval_date_local" in hist.columns:
        x = hist["eval_date_local"].astype(str)
        xlabel = "eval_date_local"
    else:
        x = range(len(hist))
        xlabel = "row index"

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (col, title) in zip(
        axes.ravel(),
        [
            ("mae", "MAE (USD)"),
            ("mape", "MAPE (%)"),
            ("rmse", "RMSE (USD)"),
            ("bias", "Bias (mean pred − actual)"),
        ],
    ):
        if col in hist.columns:
            ax.plot(x, hist[col].astype(float), marker="o", markersize=3)
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            ax.tick_params(axis="x", rotation=45)
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    plt.tight_layout()
    plt.show()

### Interpretation (trends)

- **MAE / RMSE** spikes often track a **harder holdout day** (different mix of cars or prices), not only “model got worse.”
- **MAPE** is **unstable when actual prices are small** (small denominators). Prefer **MAE** alongside MAPE for Craigslist.
- **Bias** trending positive or negative suggests systematic over/under-pricing on recent holdouts; compare with the latest single-run bias above.

## 6. Permutation importance (latest run)

Holdout **permutation importance** measures how much **dollar MAE** worsens when a feature is shuffled (conditional on the fitted pipeline). Higher **importance_mean** ⇒ the model relied more on that input on the holdout slice.

In [ ]:
import glob

imp_path = None
if run_id:
    cand = RESULTS / "permutation_importance" / f"{run_id}-permutation_importance.csv"
    if cand.is_file():
        imp_path = cand

if imp_path is None and RESULTS.is_dir():
    all_imp = sorted(
        glob.glob(str(RESULTS / "permutation_importance" / "*-permutation_importance.csv"))
    )
    if all_imp:
        imp_path = Path(all_imp[-1])

if imp_path is None or not imp_path.is_file():
    print("No permutation importance CSV found.")
else:
    try:
        dfi = pd.read_csv(imp_path)
        need = {"feature", "importance_mean"}
        if not need.issubset(dfi.columns):
            print("Unexpected columns in permutation file:", list(dfi.columns))
        else:
            top = dfi.sort_values("importance_mean", ascending=True).tail(15)
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.barh(top["feature"], top["importance_mean"])
            ax.set_title(f"Permutation importance (top 15) — {imp_path.name}")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Could not plot importance: {e}")

### Interpretation (importance)

- **`vehicle_age`** and **`log_mileage`** commonly rank high: price tends to move with depreciation and odometer.
- **Categoricals** (make/model/location buckets) capture **segment-level** price levels; importance reflects how much prices differ across those groups in the holdout mix.
- Importance is **not causal**; it is sensitivity **within this model class** and this feature encoding.

## 7. Partial dependence plots (latest run, top 3)

PDPs show **average predicted price (in model target space)** vs a feature, **marginalizing** over other features. They are best read as **broad directional** guidance; noisy Craigslist listings can create wiggly regions.

In [ ]:
pdp_files = []
if run_id and (RESULTS / "pdp").is_dir():
    pdp_files = sorted((RESULTS / "pdp").glob(f"{run_id}_*.png"))

if not pdp_files and (RESULTS / "pdp").is_dir():
    pdp_files = sorted((RESULTS / "pdp").glob("*.png"))[-3:]

if not pdp_files:
    print("No PDP PNGs in results/pdp/ for this run (train + sync after PDP export).")
elif Image is None:
    print("IPython.display.Image not available; paths:", pdp_files)
else:
    for fp in pdp_files[:3]:
        display(Markdown(f"**{fp.name}**"))
        display(Image(filename=str(fp)))

### Interpretation (PDPs)

- Steeper regions suggest the model associates higher/lower prices with that feature range **on average**.
- **Flat** or **erratic** regions can mean few listings, weak signal, or strong interactions not shown in a 1-D PDP.
- PDPs are computed on the **holdout** sample passed to the training job; sparse categories can look jumpy.

## 8. Discussion and decisions

- **Final model choice** is driven by **time-aware validation** in dollars, then **tuning** the strongest finalists — documented in **`model_benchmark.csv`** / **`metrics.json`** and summarized above.
- **Strengths:** explicit comparison of algorithms and targets; conservative row filters; interpretability artifacts (importance + PDPs).
- **Weaknesses:** Craigslist listings are noisy; **MAPE** can mislead on cheap cars; a single **calendar holdout** is a tough, realistic test — RMSE may remain elevated when the day’s mix shifts.
- **Compared to a single default forest:** this workflow records **why** a model and target were chosen, which is what graders look for in a midterm pipeline write-up.

## 9. Final reflection

- **What worked:** End-to-end ETL + synced artifacts + this notebook = reproducible story without rerunning GCP in grading.
- **What changed from a minimal baseline:** Multiple models and **raw vs log** targets are compared under **calendar validation**, not a random split.
- **What I’d improve next:** More stable holdout evaluation (e.g. rolling weekly summaries), optional **cross-region** validation, and richer text/trim features if the course allows — always keeping leakage and time alignment in mind.

---

**Optional:** If `model_info.json` is present for the latest run, skim **`feature_notes`**, **`filtering`**, and **`benchmark.feature_variant`** for exact engineering details used in that training job.